# Notebook 06 : Marché Français ───────────────

Cette analyse explore les tendances spécifiques au marché français — les comportements des consommateurs français reflètent-ils les mêmes dynamiques que celles observées à l'échelle mondiale ?

In [ ]:
# ── Afficher le heatmap synthèse ──────────────────────────────────────────────
from IPython.display import Image, display
from IPython.display import HTML

display(HTML("<style>#cell-output-area .jp-Cell-inputWrapper {display: none;}</style>"))
display(Image(filename=f"{CACHE_DIR}/06_fr_signal_heatmap.png", width=800))

### Pas tout à fait ! Les trois signaux racontent des histoires différentes selon le canal — et c'est précisément là que réside l'intérêt.

#### Observations Clés — La Beauté Française vue par la data
=========================================================

1. LA MASSE CHERCHE, LE LUXE NE CHERCHE PAS
- Les marques Mass dominent les recherches Google avec un indice moyen de 58.4, soit 17 fois plus que le Luxe (3.3).
- Ce n'est pas un signe de faiblesse du Luxe : c'est une signature de maturité.
- Le consommateur de Dior ou Guerlain n'a pas besoin de chercher — il sait déjà.

2. LE LUXE INSPIRE, LA MASSE DÉÇOIT
- Sur YouTube, le Luxe obtient le meilleur sentiment pondéré (0.471).
- La Mass tombe à quasi zéro (0.003) une fois pondéré par les likes : les commentaires les plus appréciés sont critiques, pas enthousiastes.
- Les Français aiment partager leur déception sur les grandes marques accessibles.

3. LA PHARMACIE TIENT SES PROMESSES
- Sur Beauté-Test, le Masstige obtient 0.532 — très proche du Prestige (0.557).
- Le canal pharmacie délivre une satisfaction produit qui rivalise avec des marques deux fois plus chères.

4. PRESTIGE ET MASSTIGE : UN ÉCART QUI SE RESSERRE
- Sur YouTube, l'écart de sentiment entre Prestige et Masstige est de -0.042 points.
- Sur Beauté-Test, il est de 0.025 points.
- La frontière entre ces deux tiers s'efface dans l'esprit du consommateur français.

5. TROIS SIGNAUX, ZÉRO CONSENSUS
- La Mass mène la recherche. Le Luxe mène le sentiment vidéo et les avis produits.
- Aucun tier ne domine sur tous les fronts — ce qui confirme que le marché français est segmenté non pas par prix, mais par usage, canal et moment d'achat.


## MÉTHODOLOGIE

PÉRIMÈTRE
- Période : janvier 2022 – décembre 2025
- Marché : France (geo=FR, langue=fr)
- Tiers : Luxe (4 marques), Prestige (4 marques), Masstige (5 marques), Mass (3 marques)
- Extension locale : L'Occitane (Prestige), Nuxe et Avène (Masstige) ajoutés au panier global pour refléter la spécificité du marché français.

SIGNAL 1 — TENDANCES DE RECHERCHE (Google Trends)
- Outil : pytrends, geo=FR, rééchantillonnage mensuel
- Normalisation : ancre "soin visage" injectée dans chaque batch ; scores calibrés au moment de la collecte (score / moyenne_ancre × 100)
- Agrégation : moyenne inter-keywords par tier, puis moyenne mensuelle

SIGNAL 2 — SENTIMENT YOUTUBE FR
- Collecte : API YouTube Data v3, requêtes en français, regionCode=FR
- Modèle : DistilCamemBERT (cmarkea/distilcamembert-base-sentiment) modèle natif français, sortie 1–5 étoiles convertie en score −1 → +1
- Pondération : score pondéré par le nombre de likes par commentaire (les commentaires les plus appréciés ont plus de poids)

SIGNAL 3 — AVIS BEAUTÉ-TEST
- Source : beaute-test.com — plateforme française d'avis beauté
- Résolution des marques : index complet scraté (/marques), correspondance fuzzy (SequenceMatcher) puis validation manuelle
- Agrégation : moyenne des notes étoiles pondérée par n_produits par marque, convertie en score −1 → +1 via (étoiles − 3) / 2

LIMITES
- Google Trends : indices relatifs, non absolus — comparaisons inter-tiers valides mais non comparables aux données NB03 (anchor différente)
- YouTube : volume de commentaires inégal par tier ; likes peu nombreux en FR vs EN — pondération plus bruitée
- Beauté-Test : plateforme de niche, surreprésentation possible des consommatrices engagées ; Mass sous-représenté (3 marques seulement)
- Clarins et L'Occitane sont des entreprises privées — absence de données financières comparables ; inclus pour la pertinence du signal consommateur
- NB06 est une analyse de signal consommateur, non une analyse financière

In [ ]:
# ── Cellule 1 : Imports et configuration ──────────────────

import os
import sys
import pickle
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pytrends.request import TrendReq
from dotenv import load_dotenv
load_dotenv()

# ── Chemin vers helpers_fr ────────────────────────────────
sys.path.append(os.path.abspath(".."))
from src.helpers_fr import (
    TIER_COULEURS_FR,
    TIER_ORDER_FR,
    MARQUES_FR,
    KEYWORDS_TRENDS_FR,
    KEYWORDS_YOUTUBE_FR,
    KEYWORDS_BEAUTETEST_FR,
    get_batches,
    save_chart,
)

# ── Paramètres globaux ────────────────────────────────────
GEO       = "FR"
TIMEFRAME = "2022-01-01 2025-12-31"
CACHE_DIR = "../data/processed"
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Vérification ──────────────────────────────────────────
print("Configuration chargée ✓")
print(f"Marché : {GEO} | Période : {TIMEFRAME}")
print()
for tier, marques in MARQUES_FR.items():
    print(f"  {tier:10} → {', '.join(marques)}")

In [ ]:
# ── Cellule 2 : Google Trends France — collecte des données ──────────────────
# Méthode identique à NB03 : chaque keyword est normalisé par rapport à l'ancre
# au moment de la collecte (score / moyenne_ancre × 100).
# L'ancre n'est pas stockée — seuls les scores relatifs sont conservés.

ANCHOR     = "soin visage"
CACHE_FILE = f"{CACHE_DIR}/trends_fr_raw.pkl"
USE_CACHE  = True  # changer dependant du besoin

# ── Helpers ───────────────────────────────────────────────────────────────────
import random

def fetch_batch_fr(pytrends, keywords, geo, timeframe, retries=3):
    """Collecte un batch de 4 mots-clés + ancre, normalise par rapport à l'ancre.
    Réessaie avec backoff exponentiel en cas de rate limit (429)."""
    batch = [ANCHOR] + keywords[:4]
    for attempt in range(retries):
        try:
            pytrends.build_payload(batch, geo=geo, timeframe=timeframe)
            time.sleep(1.5)
            df = pytrends.interest_over_time()
            if df.empty or ANCHOR not in df.columns:
                return None
            if "isPartial" in df.columns:
                df = df.drop(columns=["isPartial"])
            anchor_mean = df[ANCHOR].replace(0, 1).mean()
            result = {}
            for kw in keywords[:4]:
                if kw in df.columns:
                    result[kw] = (df[kw] / anchor_mean * 100).values
            return pd.DataFrame(result, index=df.index)
        except Exception as e:
            wait = (2 ** attempt) * 10 + random.uniform(0, 5)
            print(f"  ⚠ Tentative {attempt + 1}/{retries} échouée ({e}) — attente {wait:.0f}s")
            time.sleep(wait)
    print(f"  ✗ Batch abandonné après {retries} tentatives")
    return None


def fetch_tier_fr(pytrends, keywords, geo, timeframe):
    chunks = [keywords[i:i+4] for i in range(0, len(keywords), 4)]
    frames = []
    for chunk in chunks:
        df = fetch_batch_fr(pytrends, chunk, geo, timeframe)
        if df is not None:
            frames.append(df)
        time.sleep(8)  # augmenté pour éviter le 429
    if not frames:
        return None
    return pd.concat(frames, axis=1)


# ── Collecte ──────────────────────────────────────────────────────────────────
if USE_CACHE and os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        trends_fr_raw = pickle.load(f)
    print("Cache chargé ✓")

else:
    pytrends      = TrendReq(hl="fr-FR", tz=60)
    trends_fr_raw = {}

    for tier, kw_dict in KEYWORDS_TRENDS_FR.items():
        print(f"\n── {tier} ──────────────────────────────")
        all_kw = kw_dict["marques"] + kw_dict["comportement"]
        df_tier = fetch_tier_fr(pytrends, all_kw, GEO, TIMEFRAME)
        if df_tier is not None:
            df_tier.index = pd.to_datetime(df_tier.index).normalize()
            trends_fr_raw[tier] = df_tier
            print(f"  ✓ {len(df_tier)} semaines, {len(df_tier.columns)} mots-clés")
        else:
            print(f"  ⚠ Aucune donnée pour {tier}")
        time.sleep(3)

    with open(CACHE_FILE, "wb") as f:
        pickle.dump(trends_fr_raw, f)
    print(f"\nSauvegardé : trends_fr_raw.pkl ✓")

# ── Vérification ──────────────────────────────────────────────────────────────
print("\nRésumé des données collectées :")
for tier, df in trends_fr_raw.items():
    print(f"  {tier:10} → {len(df.columns)} mots-clés | "
          f"index moyen : {df.mean().mean():.1f} | "
          f"pic : {df.max().max():.1f}")

In [ ]:
# ── Cellule 3 : Agrégation par tier ──────────────────────────────────────────
# trends_fr_raw est {tier: df} — normalisation déjà effectuée en cellule 2.
# Ici on moyenne les mots-clés du tier et on rééchantillonne en mensuel.

trends_fr_tier = {}

for tier, df in trends_fr_raw.items():
    if df is None or df.empty:
        continue

    df.index = pd.to_datetime(df.index).normalize()

    # Moyenne inter-keywords → série temporelle du tier
    tier_mean    = df.mean(axis=1)
    tier_monthly = tier_mean.resample("ME").mean()

    trends_fr_tier[tier] = tier_monthly
    print(f"{tier:10} → {len(tier_monthly)} mois | "
          f"index moyen : {tier_monthly.mean():.1f} | "
          f"pic : {tier_monthly.max():.1f}")

df_trends_fr = pd.DataFrame(trends_fr_tier)
df_trends_fr.to_csv(f"{CACHE_DIR}/trends_fr_summary.csv")
print(f"\nSauvegardé : trends_fr_summary.csv ✓")

In [ ]:
# ── Cellule 4a : Visualisation — série temporelle Google Trends FR ────────────

import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#FAFAF8")
ax.set_facecolor("#FAFAF8")

for tier in TIER_ORDER_FR:
    if tier in trends_fr_tier:
        series = trends_fr_tier[tier]
        ax.plot(series.index, series.values,
                color=TIER_COULEURS_FR[tier],
                linewidth=2.2, marker="o", markersize=3,
                label=tier)

ax.axhline(100, color="#AAAAAA", lw=1.0, ls="--", alpha=0.6, label="Ancre : soin visage (100)")
ax.set_title("Intérêt de recherche Google par tier — France (2022–2025)",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Indice relatif (ancre : soin visage = 100)", fontsize=10)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", labelsize=8, rotation=45)
ax.legend(fontsize=9, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
save_chart(fig, "06_trends_fr_serie.png")
plt.show()

In [ ]:
# ── Cellule 4b : Visualisation — index moyen par tier avec min/max ────────────

import numpy as np

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor("#FAFAF8")
ax.set_facecolor("#FAFAF8")

tiers   = [t for t in TIER_ORDER_FR if t in trends_fr_tier]
means   = [trends_fr_tier[t].mean() for t in tiers]
mins    = [trends_fr_tier[t].min()  for t in tiers]
maxs    = [trends_fr_tier[t].max()  for t in tiers]
colours = [TIER_COULEURS_FR[t]      for t in tiers]
x       = np.arange(len(tiers))

ax.bar(x, means, color=colours, width=0.5, zorder=3, alpha=0.9)

for i, (mn, mx, mean) in enumerate(zip(mins, maxs, means)):
    ax.plot([x[i], x[i]], [mn, mx], color="#555555", lw=1.5, zorder=4)
    ax.plot([x[i] - 0.08, x[i] + 0.08], [mn, mn], color="#555555", lw=1.5, zorder=4)
    ax.plot([x[i] - 0.08, x[i] + 0.08], [mx, mx], color="#555555", lw=1.5, zorder=4)
    ax.text(x[i], mx + (max(maxs) * 0.02), f"{mean:.1f}",
            ha="center", fontsize=10, fontweight="bold",
            color=TIER_COULEURS_FR[tiers[i]])

ax.axhline(100, color="#AAAAAA", lw=1.0, ls="--", alpha=0.6)
ax.text(len(tiers) - 0.45, 102, "soin visage = 100",
        fontsize=8, color="#AAAAAA", va="bottom")

ax.set_xticks(x)
ax.set_xticklabels(tiers, fontsize=12, fontweight="bold")
ax.set_ylabel("Indice relatif (ancre : soin visage = 100)", fontsize=10)
ax.set_title("Index moyen de recherche par tier — France (2022–2025)\navec min / max mensuels",
             fontsize=12, fontweight="bold")
ax.set_xlim(-0.5, len(tiers) - 0.5)
ax.set_ylim(0, max(maxs) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)

plt.tight_layout(pad=2.0)      
save_chart(fig, "06_trends_fr_barchart.png")
plt.show()

In [ ]:
# ── Cellule 5 : YouTube France — collecte des commentaires ─

from googleapiclient.discovery import build
import os

YOUTUBE_API_KEY    = os.getenv("YOUTUBE_API_KEY")
CACHE_FILE_YT_FR   = f"{CACHE_DIR}/youtube_comments_fr_raw.pkl"
USE_CACHE          = True # Changer dependant du besoin

VIDEOS_PER_QUERY   = 10
COMMENTS_PER_VIDEO = 100

def get_videos(youtube, query, max_results=10):
    """Recherche des vidéos YouTube en français."""
    res = youtube.search().list(
        q=query,
        part="id,snippet",
        maxResults=max_results,
        type="video",
        relevanceLanguage="fr",
        regionCode="FR",
        order="relevance",
        publishedAfter="2020-01-01T00:00:00Z",
    ).execute()
    return [
        {
            "video_id":  item["id"]["videoId"],
            "title":     item["snippet"]["title"],
            "channel":   item["snippet"]["channelTitle"],
            "published": item["snippet"]["publishedAt"],
        }
        for item in res.get("items", [])
        if item["id"].get("videoId")
    ]

def get_comments(youtube, video_id, max_results=100):
    """Collecte les commentaires d'une vidéo."""
    comments = []
    try:
        res = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(max_results, 100),
            textFormat="plainText",
            order="relevance",
        ).execute()
        for item in res.get("items", []):
            snippet = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "text":      snippet["textDisplay"],
                "likes":     snippet["likeCount"],
                "published": snippet["publishedAt"],
            })
    except Exception as e:
        print(f"    ✗ Commentaires désactivés ou erreur : {e}")
    return comments

if USE_CACHE and os.path.exists(CACHE_FILE_YT_FR):
    with open(CACHE_FILE_YT_FR, "rb") as f:
        yt_fr_raw = pickle.load(f)
    print("Cache chargé ✓")

else:
    youtube   = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
    yt_fr_raw = {}

    for tier, queries in KEYWORDS_YOUTUBE_FR.items():
        print(f"\n── {tier} ──────────────────────────────")
        yt_fr_raw[tier] = []

        for query in queries:
            print(f"  Requête : '{query}'")
            videos = get_videos(youtube, query, max_results=VIDEOS_PER_QUERY)

            for vid in videos:
                comments = get_comments(youtube, vid["video_id"],
                                        max_results=COMMENTS_PER_VIDEO)
                vid["comments"] = comments
                yt_fr_raw[tier].append(vid)
                print(f"    ✓ {vid['title'][:50]:50} → {len(comments)} commentaires")

            time.sleep(1)

    with open(CACHE_FILE_YT_FR, "wb") as f:
        pickle.dump(yt_fr_raw, f)
    print(f"\nSauvegardé : youtube_comments_fr_raw.pkl ✓")

# ── Vérification ──────────────────────────────────────────
print("\nRésumé :")
for tier, videos in yt_fr_raw.items():
    total = sum(len(v["comments"]) for v in videos)
    print(f"  {tier:10} → {len(videos)} vidéos | {total} commentaires")

In [ ]:
# ── Cellule 6 : Sentiment DistilCamemBERT ─────────────────

from transformers import pipeline

CACHE_FILE_SENT_FR = f"{CACHE_DIR}/youtube_sentiment_fr.pkl"
USE_CACHE          = True # Changer dependant du besoin

if USE_CACHE and os.path.exists(CACHE_FILE_SENT_FR):
    with open(CACHE_FILE_SENT_FR, "rb") as f:
        sentiment_fr_raw = pickle.load(f)
    print("Cache chargé ✓")

else:
    print("Chargement du modèle DistilCamemBERT...")
    analyzer = pipeline(
        task="text-classification",
        model="cmarkea/distilcamembert-base-sentiment",
        tokenizer="cmarkea/distilcamembert-base-sentiment",
        top_k=None,
    )
    print("Modèle chargé ✓\n")

    def camembert_to_compound(scores):
        """Convertit les probabilités 1-5 étoiles en score -1/+1."""
        star_map = {"1 star": 1, "2 stars": 2, "3 stars": 3,
                    "4 stars": 4, "5 stars": 5}
        weighted = sum(star_map[s["label"]] * s["score"] for s in scores)
        return (weighted - 3) / 2  # midpoint 3 = 0.0

    def score_text(text, max_length=512):
        """Score un texte — tronquer si trop long."""
        try:
            text    = str(text).strip()[:max_length]
            results = analyzer(text)
            return camembert_to_compound(results[0])
        except Exception:
            return None

    sentiment_fr_raw = {}

    for tier, videos in yt_fr_raw.items():
        print(f"── {tier} ──────────────────────────────")
        tier_scores = []

        for vid in videos:
            for comment in vid["comments"]:
                text  = comment.get("text", "")
                likes = comment.get("likes", 0)
                if not text:
                    continue
                score = score_text(text)
                if score is not None:
                    tier_scores.append({
                        "tier":     tier,
                        "video_id": vid["video_id"],
                        "text":     text,
                        "likes":    likes,
                        "compound": score,
                    })

        sentiment_fr_raw[tier] = tier_scores
        print(f"  {len(tier_scores)} commentaires scorés")

    with open(CACHE_FILE_SENT_FR, "wb") as f:
        pickle.dump(sentiment_fr_raw, f)
    print(f"\nSauvegardé : youtube_sentiment_fr.pkl ✓")

# ── Vérification rapide ───────────────────────────────────
print("\nAperçu des scores :")
for tier, scores in sentiment_fr_raw.items():
    if scores:
        compounds = [s["compound"] for s in scores]
        print(f"  {tier:10} → {len(compounds)} scores | "
              f"moyenne : {sum(compounds)/len(compounds):.3f} | "
              f"min : {min(compounds):.3f} | max : {max(compounds):.3f}")

In [ ]:
# ── Cellule 7 : Agrégation par tier et sauvegarde ─────────

records = []
for tier, scores in sentiment_fr_raw.items():
    for s in scores:
        records.append({
            "tier":     tier,
            "compound": s["compound"],
            "likes":    s["likes"],
        })

df_sent_fr = pd.DataFrame(records)

# ── Score pondéré par likes (même approche que NB04-A) ────
df_sent_fr["weight"] = df_sent_fr["likes"].clip(lower=1)

youtube_tier_sentiment_fr = (
    df_sent_fr.groupby("tier")
    .apply(lambda g: pd.Series({
        "Weighted_Sentiment": (
            (g["compound"] * g["weight"]).sum() / g["weight"].sum()
        ),
        "Mean_Sentiment": g["compound"].mean(),
        "Comment_Count":  len(g),
    }))
    .reindex(TIER_ORDER_FR)
    .reset_index()
)

print(youtube_tier_sentiment_fr.to_string(index=False))

youtube_tier_sentiment_fr.to_csv(
    f"{CACHE_DIR}/youtube_sentiment_fr_tier.csv", index=False
)
print("\nSauvegardé : youtube_sentiment_fr_tier.csv ✓")

In [ ]:
# ── Cellule 8 : Visualisation — sentiment YouTube FR ──────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sentiment YouTube — Marché Français (DistilCamemBERT)",
             fontsize=13, fontweight="bold")

# ── Panel 1 : Diverging dot plot — score pondéré ─────────
ax1 = axes[0]
for i, tier in enumerate(TIER_ORDER_FR):
    val = tier_sentiment_fr.loc[
        tier_sentiment_fr["tier"] == tier, "Weighted_Sentiment"
    ].values[0]
    color = TIER_COULEURS_FR[tier]
    ax1.plot([0, val], [i, i], color=color, linewidth=2.5, alpha=0.7)
    ax1.scatter(val, i, color=color, s=180, zorder=5)
    ax1.text(val + 0.03, i, f"{val:.3f}",
             va="center", fontsize=10, fontweight="bold")

ax1.axvline(0, color="grey", linewidth=1, linestyle="--", alpha=0.5)
ax1.set_yticks(range(len(TIER_ORDER_FR)))
ax1.set_yticklabels(TIER_ORDER_FR, fontsize=11)
ax1.invert_yaxis()
ax1.set_title("Score pondéré (likes)", fontsize=11, fontweight="bold")
ax1.set_xlabel("Score composé (-1 à +1)", fontsize=9)
ax1.set_xlim(-0.15, 0.6)
ax1.spines[["top", "right", "left"]].set_visible(False)
ax1.tick_params(left=False)
ax1.grid(axis="x", linestyle="--", alpha=0.3)

# ── Panel 2 : Dumbbell — pondéré vs non pondéré ───────────
ax2 = axes[1]
W_COLOR  = "#4A7C9E"
UW_COLOR = "#C4956A"

for i, tier in enumerate(TIER_ORDER_FR):
    w_val  = youtube_tier_sentiment_fr.loc[
        youtube_tier_sentiment_fr["tier"] == tier, "Weighted_Sentiment"
    ].values[0]
    uw_val = youtube_tier_sentiment_fr.loc[
        youtube_tier_sentiment_fr["tier"] == tier, "Mean_Sentiment"
    ].values[0]
    delta  = w_val - uw_val
    mid    = (w_val + uw_val) / 2

    ax2.plot([w_val, uw_val], [i, i],
             color="lightgrey", linewidth=2.5, zorder=1)
    ax2.scatter(w_val,  i, color=W_COLOR,  s=160, zorder=5)
    ax2.scatter(uw_val, i, color=UW_COLOR, s=160, zorder=5)
    ax2.text(mid, i + 0.22, f"Δ{delta:+.3f}",
             va="bottom", ha="center", fontsize=8, color="grey")

ax2.axvline(0, color="grey", linewidth=1, linestyle="--", alpha=0.5)
ax2.set_yticks(range(len(TIER_ORDER_FR)))
ax2.set_yticklabels(TIER_ORDER_FR, fontsize=11)
ax2.invert_yaxis()
ax2.set_title("Pondéré vs Non pondéré", fontsize=11, fontweight="bold")
ax2.set_xlabel("Score composé (-1 à +1)", fontsize=9)
ax2.set_xlim(-0.15, 0.6)
ax2.spines[["top", "right", "left"]].set_visible(False)
ax2.tick_params(left=False)
ax2.grid(axis="x", linestyle="--", alpha=0.3)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=W_COLOR, markersize=9, label="Pondéré (likes)"),
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=UW_COLOR, markersize=9, label="Non pondéré"),
]
ax2.legend(handles=legend_elements, frameon=False,
           fontsize=9, loc="lower right")

plt.tight_layout()
save_chart(fig, "06_sentiment_youtube_fr.png")
plt.show()

In [ ]:
# ── Cell 9 | Beauté-Test — Brand Resolution & Scraper ─────────────────────────
# Phase 1 : build index | Phase 2 : resolve | Phase 3 : scrape ratings
# ──────────────────────────────────────────────────────────────────────────────

import time, re, unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup
from difflib import SequenceMatcher
from pathlib import Path
import sys
sys.path.append("../src")
from helpers_fr import TIER_ORDER_FR, TIER_COULEURS_FR

USE_CACHE        = True # Changer dependant du besoin
CACHE_BRANDS     = "../data/processed/beautetest_brands.pkl"
CACHE_RATINGS    = "../data/processed/beautetest_ratings.pkl"
BASE_URL         = "https://www.beaute-test.com"
HEADERS          = {"User-Agent": "Mozilla/5.0"}

# ── Search terms per tier (human-readable, not slugs) ─────────────────────────
SEARCH_TERMS = {
    "Luxe":      ["Dior", "Guerlain", "Givenchy", "Yves Saint Laurent"],
    "Prestige":  ["Lancôme", "Clarins", "Giorgio Armani", "L'Occitane"],
    "Masstige":  ["La Roche-Posay", "Vichy", "CeraVe", "Nuxe", "Avène"],
    "Mass":      ["L'Oréal Paris", "Garnier", "Gemey-Maybelline"],
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def normalize(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"[''&]", " ", s)
    s = re.sub(r"[-_/]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

# ── Phase 1 : Brand index ─────────────────────────────────────────────────────
if USE_CACHE and Path(CACHE_BRANDS).exists():
    brand_index = pd.read_pickle(CACHE_BRANDS)
    print(f"[cache] brand_index — {len(brand_index)} marques")
else:
    r    = requests.get(f"{BASE_URL}/marques", headers=HEADERS, timeout=20)
    soup = BeautifulSoup(r.text, "html.parser")
    rows = [
        {"brand_name": a.get_text(strip=True), "brand_path": a["href"]}
        for a in soup.select('a[href^="/marque/"]')
        if a.get_text(strip=True)
    ]
    brand_index = (
        pd.DataFrame(rows)
          .drop_duplicates("brand_path")
          .assign(brand_norm=lambda d: d["brand_name"].map(normalize))
          .reset_index(drop=True)
    )
    brand_index.to_pickle(CACHE_BRANDS)
    print(f"[fetch] brand_index — {len(brand_index)} marques")

# ── Phase 2 : Resolve search terms → brand paths ─────────────────────────────
def resolve(term: str, index: pd.DataFrame) -> dict:
    norm = normalize(term)
    exact = index[index["brand_norm"] == norm]
    if not exact.empty:
        row = exact.iloc[0]
        return {"brand_name": row.brand_name, "brand_path": row.brand_path, "score": 1.0}
    scored = index.assign(score=index["brand_name"].map(lambda x: similarity(term, x)))
    best   = scored.nlargest(1, "score").iloc[0]
    return {"brand_name": best.brand_name, "brand_path": best.brand_path, "score": round(best.score, 3)}

resolved = pd.DataFrame([
    {"tier": tier, "search_term": term, **resolve(term, brand_index)}
    for tier, terms in SEARCH_TERMS.items()
    for term in terms
])

print("\nResolution map:")
print(resolved[["tier", "search_term", "brand_name", "brand_path", "score"]].to_string(index=False))
assert (resolved["score"] >= 0.85).all(), "⚠ Low-confidence match — review before scraping"

# ── Phase 3 : Scrape aggregate ratings ───────────────────────────────────────
def scrape_brand_rating(brand_path: str) -> dict:
    """Return mean star rating from brand page product cards."""
    url  = f"{BASE_URL}{brand_path}"
    r    = requests.get(url, headers=HEADERS, timeout=20)
    soup = BeautifulSoup(r.text, "html.parser")

    ratings = []
    for note in soup.select("span.note, div.note, [class*='note']"):
        txt = note.get_text(strip=True).replace(",", ".")
        try:
            val = float(re.search(r"\d+\.?\d*", txt).group())
            if 0 < val <= 5:
                ratings.append(val)
        except (AttributeError, ValueError):
            continue

    return {
        "n_products": len(ratings),
        "mean_stars":  round(sum(ratings) / len(ratings), 3) if ratings else None,
    }

if USE_CACHE and Path(CACHE_RATINGS).exists():
    ratings_df = pd.read_pickle(CACHE_RATINGS)
    print("\n[cache] ratings loaded")
else:
    records = []
    for _, row in resolved.iterrows():
        stats = scrape_brand_rating(row.brand_path)
        records.append({
            "tier":       row.tier,
            "brand":      row.brand_name,
            "brand_path": row.brand_path,
            **stats,
        })
        print(f"  {row.brand_name:30} n={stats['n_products']:3}  stars={stats['mean_stars']}")
        time.sleep(2)

    ratings_df = pd.DataFrame(records)
    ratings_df.to_pickle(CACHE_RATINGS)

# ── Tier aggregation ──────────────────────────────────────────────────────────
tier_ratings = (
    ratings_df
    .dropna(subset=["mean_stars"])
    .groupby("tier")
    .agg(mean_stars=("mean_stars", "mean"), n_brands=("brand", "count"))
    .reindex(TIER_ORDER_FR)
    .reset_index()
)

print("\nTier aggregation:")
print(tier_ratings.to_string(index=False))

In [ ]:
# ── Cell 10 | Beauté-Test — Sentiment Conversion & Visualisation ──────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import sys
sys.path.append("../src")
from helpers_fr import TIER_ORDER_FR, TIER_COULEURS_FR, save_chart

# ── Sentiment conversion : (stars - 3) / 2 → [-1, +1] ───────────────────────
ratings_df["sentiment"] = (ratings_df["mean_stars"] - 3) / 2

tier_sentiment = (
    ratings_df
    .dropna(subset=["sentiment"])
    .groupby("tier")
    .apply(lambda g: pd.Series({
        "sentiment":        np.average(g["sentiment"], weights=g["n_products"]),
        "n_products_total": g["n_products"].sum(),
        "n_brands":         len(g)
    }))
    .reindex(TIER_ORDER_FR)
    .reset_index()
)

print("Beauté-Test sentiment par tier (pondéré par n_products):")
print(tier_sentiment.to_string(index=False))

# ── Visualisation : diverging dot plot (tier level) ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle("Beauté-Test FR — Sentiment consommateurs par tier",
             fontsize=13, fontweight="bold")

colours = [TIER_COULEURS_FR[t] for t in tier_sentiment["tier"]]

ax.axvline(0, color="grey", lw=0.8, ls="--")
for i, row in tier_sentiment.iterrows():
    ax.plot([0, row.sentiment], [i, i], color=colours[i], lw=2.5)
    ax.scatter(row.sentiment, i, color=colours[i], s=120, zorder=5)
    ax.text(row.sentiment + 0.03, i,
            f"{row.sentiment:.3f}  (n={int(row.n_products_total)})",
            va="center", fontsize=10)

ax.set_yticks(range(len(tier_sentiment)))
ax.set_yticklabels(tier_sentiment["tier"], fontsize=11)
ax.invert_yaxis()
ax.set_xlabel("Score sentiment (−1 → +1)", fontsize=10)
ax.set_xlim(0.0, 1.0)

patches = [mpatches.Patch(color=TIER_COULEURS_FR[t], label=t) for t in TIER_ORDER_FR]
ax.legend(handles=patches, loc="lower right", fontsize=10)

plt.tight_layout()
save_chart(fig, "06_sentiment_beautetest_fr.png")
plt.show()

# ── Dynamic observation ───────────────────────────────────────────────────────
top_tier     = tier_sentiment.loc[tier_sentiment.sentiment.idxmax(), "tier"]
top_score    = tier_sentiment.sentiment.max()
bottom_tier  = tier_sentiment.loc[tier_sentiment.sentiment.idxmin(), "tier"]
bottom_score = tier_sentiment.sentiment.min()
spread       = round(top_score - bottom_score, 3)

print(f"""
Observations :
- {top_tier} leads Beauté-Test sentiment ({top_score:.3f})
- {bottom_tier} trails at {bottom_score:.3f}
- Tier spread : {spread} — {'tight' if spread < 0.2 else 'moderate' if spread < 0.35 else 'wide'}
- Monotonic descent confirmed : {list(tier_sentiment['tier']) == TIER_ORDER_FR}
""")

In [ ]:
# ── Cell 11 | NB06 Synthesis — FR Signal Heatmap (LinkedIn) ───────────────────

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import sys
sys.path.append("../src")
from helpers_fr import TIER_ORDER_FR, save_chart

# ── Scores live depuis les DataFrames ─────────────────────
yt_df = pd.read_csv(f"{CACHE_DIR}/youtube_sentiment_fr_tier.csv")
yt_scores = yt_df.set_index("tier")["Weighted_Sentiment"]
bt_scores = tier_sentiment.set_index("tier")["sentiment"]
trends_scores = pd.Series({t: trends_fr_tier[t].mean() for t in TIER_ORDER_FR})

# ── Raw scores (from prior cells) ─────────────────────────────────────────────
raw = {
    "Tendances\nde recherche": trends_scores.to_dict(),
    "Sentiment\nYouTube FR":   yt_scores.to_dict(),
    "Avis\nBeauté-Test":       bt_scores.to_dict(),
}

SOURCES = list(raw.keys())

# ── Build matrix & min-max normalise per source ───────────────────────────────
matrix_raw  = np.array([[raw[s][t] for s in SOURCES] for t in TIER_ORDER_FR], dtype=float)
matrix_norm = np.zeros_like(matrix_raw)
for j in range(matrix_raw.shape[1]):
    col = matrix_raw[:, j]
    lo, hi = col.min(), col.max()
    matrix_norm[:, j] = (col - lo) / (hi - lo) if hi > lo else np.zeros_like(col)

# ── Colour map : amber → green ────────────────────────────────────────────────
cmap = mcolors.LinearSegmentedColormap.from_list(
    "fr_signal", ["#E8C547", "#5A8C5A"]
)

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 7))
fig.patch.set_facecolor("#FAFAF8")

ax = fig.add_axes([0.18, 0.20, 0.75, 0.60])
ax.set_facecolor("#FAFAF8")

n_tiers, n_sources = matrix_norm.shape

for i in range(n_tiers):
    for j in range(n_sources):
        val     = matrix_norm[i, j]
        raw_val = matrix_raw[i, j]
        colour  = cmap(val)

        rect = plt.Rectangle([j, i], 1, 1, color=colour, ec="white", lw=2.5)
        ax.add_patch(rect)

        text_colour = "white" if val > 0.55 else "#2A2A2A"

        # Raw score — large, bold
        ax.text(j + 0.5, i + 0.52, f"{raw_val:.3f}",
                ha="center", va="center", fontsize=12,
                fontweight="bold", color=text_colour)

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_xlim(0, n_sources)
ax.set_ylim(0, n_tiers)
ax.invert_yaxis()

ax.set_xticks([j + 0.5 for j in range(n_sources)])
ax.set_xticklabels([""] * n_sources)

# ── Column headers with scale description ─────────────────────────────────────
SOURCE_LABELS = [
    "Tendances\nde recherche",
    "Sentiment\nYouTube FR",
    "Avis\nBeauté-Test"
]
SOURCE_DESCS = [
    "ancre 'soin visage' = 100",
    "échelle −1 → +1 (pondéré)",
    "échelle −1 → +1 (étoiles)"
]
for j, (label, desc) in enumerate(zip(SOURCE_LABELS, SOURCE_DESCS)):
    ax.text(j + 0.5, 1.00, label,
            ha="center", va="bottom", fontsize=11,
            fontweight="bold", color="#2A2A2A",
            transform=ax.get_xaxis_transform())
    ax.text(j + 0.5, -0.03, desc,
            ha="center", va="bottom", fontsize=7.5,
            color="#777777", style="italic",
            transform=ax.get_xaxis_transform())

ax.set_yticks([i + 0.5 for i in range(n_tiers)])
ax.set_yticklabels(TIER_ORDER_FR, fontsize=13, fontweight="bold", color="#2A2A2A")
ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

# ── Title ─────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.93,
         "Trois signaux, un marché — La beauté française vue par la data",
         ha="center", fontsize=13, fontweight="bold", color="#1A1A1A")

fig.text(0.5, 0.895,
         "Tendances de recherche (Google Trends)  ·  Sentiment YouTube FR  ·  Avis Beauté-Test",
         ha="center", fontsize=9, color="#555555")

# ── Colour bar ────────────────────────────────────────────────────────────────
cbar_ax = fig.add_axes([0.18, 0.13, 0.75, 0.022])
norm    = mcolors.Normalize(vmin=0, vmax=1)
sm      = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cb.set_ticks([0, 0.5, 1])
cb.set_ticklabels(["Signal faible", "Moyen", "Signal fort"], fontsize=9, color="#555555")
cb.outline.set_visible(False)

# ── Bottom legend — explain the two numbers ───────────────────────────────────
plt.savefig("../data/processed/06_fr_signal_heatmap.png",
            dpi=180, bbox_inches="tight", facecolor="#FAFAF8")
save_chart(fig, "06_fr_signal_heatmap.png")
plt.show()

print("Saved → 06_fr_signal_heatmap.png")

In [ ]:
# ── Cell 12 | YouTube FR vs Beauté-Test — Dumbbell Plot ───────────────────────

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import sys
sys.path.append("../src")
from helpers_fr import TIER_ORDER_FR, TIER_COULEURS_FR, save_chart

# ── Merge YouTube and Beauté-Test tier scores ─────────────────────────────────
yt = yt_scores.rename("YouTube FR")
bt = bt_scores.rename("Beauté-Test")

dumbbell = pd.DataFrame([yt, bt]).T.reindex(TIER_ORDER_FR).reset_index()
dumbbell.columns = ["tier", "yt", "bt"]
dumbbell["delta"] = (dumbbell["bt"] - dumbbell["yt"]).round(3)

print(dumbbell.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor("#FAFAF8")
ax.set_facecolor("#FAFAF8")

ax.axvline(0, color="#CCCCCC", lw=0.8, ls="--")

for i, row in dumbbell.iterrows():
    colour = TIER_COULEURS_FR[row.tier]

    # Connecting line
    ax.plot([row.yt, row.bt], [i, i], color=colour, lw=2.5, alpha=0.7)

    # YouTube dot
    ax.scatter(row.yt, i, color=colour, s=120, zorder=5, marker="o")

    # Beauté-Test dot
    ax.scatter(row.bt, i, color=colour, s=120, zorder=5, marker="D")

    # Delta label — above midpoint
    mid = (row.yt + row.bt) / 2
    sign = "+" if row.delta > 0 else ""
    ax.text(mid, i - 0.09, f"Δ {sign}{row.delta:.3f}",
            ha="center", va="center", fontsize=8.5, color="#555555")

    # Score labels at endpoints
    ax.text(row.yt - 0.012, i + 0.19, f"{row.yt:.3f}",
            ha="center", fontsize=8, color=colour)
    ax.text(row.bt + 0.012, i + 0.19, f"{row.bt:.3f}",
            ha="center", fontsize=8, color=colour)

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_yticks(range(len(dumbbell)))
ax.set_yticklabels(dumbbell["tier"], fontsize=12, fontweight="bold")
ax.invert_yaxis()
ax.set_xlabel("Score sentiment (−1.0 → +1.0)", fontsize=10)
ax.set_xlim(-0.1, 1.0)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)

# ── Legend ────────────────────────────────────────────────────────────────────
yt_dot = mlines.Line2D([], [], color="#888888", marker="o", ms=8,
                        linestyle="None", label="Sentiment YouTube FR")
bt_dot = mlines.Line2D([], [], color="#888888", marker="D", ms=8,
                        linestyle="None", label="Avis Beauté-Test")
ax.legend(handles=[yt_dot, bt_dot], loc="lower right", fontsize=9,
          frameon=True, framealpha=0.6, edgecolor="#CCCCCC")

# ── Title ─────────────────────────────────────────────────────────────────────
ax.set_title("YouTube FR vs Beauté-Test — Sentiment par tier",
             fontsize=12, fontweight="bold", pad=14, color="#1A1A1A")
ax.text(0.5, 1.01,
        "Δ positif = les avis produits surpassent le sentiment vidéo",
        transform=ax.transAxes, ha="center", fontsize=8.5,
        color="#666666", style="italic")

plt.tight_layout()
save_chart(fig, "06_dumbbell_yt_vs_beautetest.png")
plt.show()

print("Saved → 06_dumbbell_yt_vs_beautetest.png")

In [ ]:
# ── Cellule 13 : Observations clés — Marché Français ─────────────────────────

# ── Extraire les chiffres clés depuis les DataFrames ─────────────────────────
mass_trends   = trends_fr_tier["Mass"].mean()
luxe_trends   = trends_fr_tier["Luxe"].mean()
ratio_trends  = mass_trends / luxe_trends
if "youtube_sentiment_fr_tier" not in dir():
    youtube_sentiment_fr_tier = pd.read_csv(f"{CACHE_DIR}/youtube_sentiment_fr_tier.csv")

luxe_yt       = youtube_sentiment_fr_tier.loc[youtube_sentiment_fr_tier["tier"] == "Luxe",   "Weighted_Sentiment"].values[0]
mass_yt       = youtube_sentiment_fr_tier.loc[youtube_sentiment_fr_tier["tier"] == "Mass",   "Weighted_Sentiment"].values[0]
prestige_yt   = youtube_sentiment_fr_tier.loc[youtube_sentiment_fr_tier["tier"] == "Prestige", "Weighted_Sentiment"].values[0]
masstige_yt   = youtube_sentiment_fr_tier.loc[youtube_sentiment_fr_tier["tier"] == "Masstige", "Weighted_Sentiment"].values[0]
delta_yt      = round(prestige_yt - masstige_yt, 3)

luxe_bt       = tier_sentiment.loc[tier_sentiment["tier"] == "Luxe",     "sentiment"].values[0]
prestige_bt   = tier_sentiment.loc[tier_sentiment["tier"] == "Prestige", "sentiment"].values[0]
masstige_bt   = tier_sentiment.loc[tier_sentiment["tier"] == "Masstige", "sentiment"].values[0]
mass_bt       = tier_sentiment.loc[tier_sentiment["tier"] == "Mass",     "sentiment"].values[0]
delta_bt      = round(prestige_bt - masstige_bt, 3)

observations = f"""
OBSERVATIONS CLÉS — LA BEAUTÉ FRANÇAISE VUE PAR LA DATA
=========================================================

1. LA MASSE CHERCHE, LE LUXE NE CHERCHE PAS
   • Les marques Mass dominent les recherches Google avec un indice moyen de {mass_trends:.1f},
     soit {ratio_trends:.0f} fois plus que le Luxe ({luxe_trends:.1f}).
   • Ce n'est pas un signe de faiblesse du Luxe : c'est une signature de maturité.
   • Le consommateur de Dior ou Guerlain n'a pas besoin de chercher — il sait déjà.

2. LE LUXE INSPIRE, LA MASSE DÉÇOIT
   • Sur YouTube, le Luxe obtient le meilleur sentiment pondéré ({luxe_yt:.3f}).
   • La Mass tombe à quasi zéro ({mass_yt:.3f}) une fois pondéré par les likes :
     les commentaires les plus appréciés sont critiques, pas enthousiastes.
   • Les Français aiment partager leur déception sur les grandes marques accessibles.

3. LA PHARMACIE TIENT SES PROMESSES
   • Sur Beauté-Test, le Masstige obtient {masstige_bt:.3f} — très proche
     du Prestige ({prestige_bt:.3f}).
   • Le canal pharmacie délivre une satisfaction produit qui rivalise
     avec des marques deux fois plus chères.

4. PRESTIGE ET MASSTIGE : UN ÉCART QUI SE RESSERRE
   • Sur YouTube, l'écart de sentiment entre Prestige et Masstige est de {delta_yt:.3f} points.
   • Sur Beauté-Test, il est de {delta_bt:.3f} points.
   • La frontière entre ces deux tiers s'efface dans l'esprit du consommateur français.

5. TROIS SIGNAUX, ZÉRO CONSENSUS
   • La Mass mène la recherche. Le Luxe mène le sentiment vidéo et les avis produits.
   • Aucun tier ne domine sur tous les fronts — ce qui confirme que le marché français
     est segmenté non pas par prix, mais par usage, canal et moment d'achat.
"""

print(observations)

with open(f"{CACHE_DIR}/06_observations_cles.txt", "w", encoding="utf-8") as f:
    f.write(observations)
print("Sauvegardé : 06_observations_cles.txt ✓")